# step6 — 처방: 값 조향 vs Spotlight (헤드라인 비교)

**어느 스텝·어느 RQ:** step6(처방). step1~5가 "왜 값(내용)이냐"를 밝혔고, 여기서 **직접 손대서** 확인한다.

**무엇을 확인하나.**
출발점은 **절벽 바닥** — 지침은 camelCase인데 앞 코드 12개가 전부 snake다(step1에서 준수율 0).
여기에 네 가지를 걸고 **얼마나 되살아나는지** 잰다.

| 방법 | 무엇을 하나 | 예상 |
|---|---|---|
| 무개입 | 아무것도 안 함 | 바닥 (하한선) |
| **값 조향 @ 맞는 층** | 그 층 잔차에 `세기 × (camel−snake 방향)` 더하기 | **회복** |
| 값 조향 @ 엉뚱한 층 | 초반 층에 같은 방향 | 거의 안 됨 (층 특이성) |
| **Spotlight** | 지침 어텐션 비중을 ψ까지 끌어올림 | 거의 안 됨 (어텐션 접근 반증) |

**Spotlight는 두 가지 스팬으로 돌린다.**
- `rule_word` : 규칙문 지시어만 — step5에서 값을 바꾼 자리와 **정확히 같은 자리**
- `instruction` : 지침 문장 전체 — 원 논문 정의

어느 쪽이 유리한지 자명하지 않다(같은 ψ면 스팬이 짧을수록 토큰당 더 세게 밀린다).
**둘 다 돌려서 "스팬 선택 탓"이라는 반론을 막는다.**

**결과가 어느 쪽으로 나오든 무슨 뜻인지 미리 밝힌다.**

| 나오는 그림 | 뜻 |
|---|---|
| 값 조향만 회복, Spotlight는 안 됨 | 우리 주장 확정. 손잡이는 내용 쪽이다 |
| Spotlight도 회복 | **어텐션 접근도 통한다** — 반박 논지를 접고 그대로 보고한다 |
| 둘 다 안 됨 | 조향 세기·층 선택을 다시 본다(세기 스윕이 그래서 있다) |
| 엉뚱한 층도 회복 | 층 특이성 주장을 접는다 |

**중요.** Spotlight 결과에는 **개입 전·후 어텐션 비중**이 함께 저장된다.
"구현을 못해서 진 것"이라는 반박을 막기 위한 증거다. 비중이 실제로 올랐는데도 회복이 없어야 근거가 된다.

**모델 하나씩.** 셀 ④ `PICK` → ⑤ 실행. 끝나면 `PICK` 바꿔 반복. 끊겨도 저장된 건 건너뛴다.
**부하:** deepseek-6.7b는 T4에서 빠듯하다. 마지막에 돌리고, 필요하면 런타임 재시작.

In [ ]:
# ② 환경 — 설치, GPU 확인, 무작위값 42 고정
!pip install -q -r requirements.txt

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('무작위값 고정:', SEED)

In [ ]:
# ③ 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
BRANCH = 'integration/step1-5'
!git fetch --quiet origin $BRANCH
!git checkout $BRANCH
!git pull --quiet origin $BRANCH
!pip install -e . -q
import sys; sys.path.insert(0, 'src')
print('브랜치:', BRANCH)

In [ ]:
# ④ 조건 설정 — 모델 하나, 절벽 바닥에서 네 가지 방법
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]
# step3에서 확인된 코드 인과 봉우리 층 (docs/step3/results.md)
PEAK  = {'qwen': 25, 'deepseek': 20, 'llama': 15, 'stability': 18}
# 대조용 '엉뚱한 층' — 초반. 같은 방향을 같은 세기로 밀어도 안 되는지 본다
EARLY = {'qwen': 5,  'deepseek': 5,  'llama': 3,  'stability': 4}

# ★ 이번에 돌릴 모델 하나 (0=qwen, 1=deepseek, 2=llama, 3=stable)
PICK = 0
MODEL = MODELS[PICK]
print('이번 모델:', MODEL.family, '| 맞는 층 L', PEAK[MODEL.family], '| 엉뚱한 층 L', EARLY[MODEL.family])

BLOCKS     = list(range(42))
STRENGTHS  = [1.0, 2.0, 4.0, 8.0]       # 조향 세기 스윕
PSI        = [0.1, 0.3]                  # Spotlight 목표 비중 (원문 기본 0.1 + 강한 값)
SPANS      = ['rule_word', 'instruction']

def cliff(block):
    """절벽 바닥 — 지침은 camel, 앞 코드 12개가 전부 위반(snake)."""
    return PrecedingCode(n_compliant=0, n_functions=12,
                         composition=Composition.POOL, pool_block=block)

INSTR = Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL)

conditions = []
for block in BLOCKS:
    pre = cliff(block)
    base = dict(model=MODEL, preceding=pre, instruction=INSTR, seed=SEED, tag='cliff')
    # 하한선
    conditions.append(Condition(**base, intervention=Intervention()))
    # 값 조향 — 맞는 층 / 엉뚱한 층
    for layer_name, layer in (('peak', PEAK[MODEL.family]), ('early', EARLY[MODEL.family])):
        for s in STRENGTHS:
            conditions.append(Condition(**base, intervention=Intervention(
                kind=InterventionKind.VALUE_ADD, layers=[layer],
                strength=s, steer_source='code_contrast')))
    # Spotlight — 스팬 2종 × ψ 2종
    for span in SPANS:
        for psi in PSI:
            conditions.append(Condition(**base, intervention=Intervention(
                kind=InterventionKind.ATTENTION_AMPLIFY, layers='all',
                amplify=psi, span=span)))

print('조건 수:', len(conditions),
      f'(묶음 {len(BLOCKS)} × [무개입 1 + 조향 {2*len(STRENGTHS)} + Spotlight {len(SPANS)*len(PSI)}])')
assert len({c.slug() for c in conditions}) == len(conditions), '파일명이 겹친다'

In [ ]:
# ⑤ 실행 — 조건마다 즉시 저장(재개)
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
import numpy as np
from collections import defaultdict

STEP = 'step6_steer'
todo = [c for c in conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 전체 {len(conditions)} / 남은 {len(todo)}')

if todo:
    handle = load_model(MODEL)
    print(f'  층수 {handle.num_layers}')
    seen = defaultdict(list)
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle, mode='steer')   # 무개입도 같은 교사강제 점수로 잰다
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ2/RQ3',
                                 prediction='값 조향은 회복, Spotlight는 거의 회복 없음'))
        ex = out.metrics.extra
        if not ex['undecidable'] and ex['recovery'] is not None:
            label = ex['method']
            if label == 'value_add':
                label += f"@L{ex['layer']}·세기{ex['strength']:g}"
            elif label == 'attn_amplify':
                label += f"·{ex['span']}·ψ{ex['psi_target']:g}"
            seen[label].append(ex['recovery'])
        del out
        if i % 40 == 0 or i == len(todo):
            print(f'  [{i}/{len(todo)}] ' + ' | '.join(
                f'{k} {np.mean(v):.3f}' for k, v in sorted(seen.items())))
    del handle
    import torch, gc; gc.collect(); torch.cuda.empty_cache()
print('완료')

In [ ]:
# ⑥ 결과 로드
from harness import result_path
from harness.results import load_result
recs = [load_result(result_path(c, step='step6_steer')) for c in conditions
        if result_path(c, step='step6_steer').exists()]
print('불러온 조건:', len(recs))

In [ ]:
# ⑦ 요약 — 헤드라인 비교표 + Spotlight가 실제로 걸렸는지
import numpy as np
from collections import defaultdict

rec_by = defaultdict(list); psi_by = defaultdict(lambda: defaultdict(list))
n_und = 0
for r in recs:
    ex = r.metrics.extra
    if ex.get('undecidable') or ex.get('recovery') is None:
        n_und += 1; continue
    m = ex['method']
    if m == 'none':
        key = '무개입 (하한선)'
    elif m == 'value_add':
        which = '맞는 층' if ex['layer'] == PEAK[MODEL.family] else '엉뚱한 층'
        key = f"값 조향 {which} L{ex['layer']} · 세기 {ex['strength']:g}"
    else:
        key = f"Spotlight {ex['span']} · ψ {ex['psi_target']:g}"
        psi_by[key]['before'].append(ex['attn_span_before'])
        psi_by[key]['after'].append(ex['attn_span_after'])
    rec_by[key].append(ex['recovery'])

print(f'[{MODEL.family}] 판정 불가로 뺀 조건 {n_und}개\n')
print(f"{'방법':<34}{'준수 회복':>12}{'±95%':>9}{'n':>6}")
for k in sorted(rec_by, key=lambda x: (not x.startswith('무개입'), x)):
    v = rec_by[k]
    ci = 1.96*np.std(v, ddof=1)/np.sqrt(len(v)) if len(v) > 1 else 0.0
    print(f'{k:<34}{np.mean(v):>12.3f}{ci:>9.3f}{len(v):>6}')

if psi_by:
    print('\nSpotlight가 실제로 걸렸는지 (개입 전 → 후 어텐션 비중)')
    print(f"{'조건':<34}{'개입 전':>10}{'개입 후':>10}{'올랐나':>9}")
    for k in sorted(psi_by):
        b, a = np.mean(psi_by[k]['before']), np.mean(psi_by[k]['after'])
        print(f"{k:<34}{b:>10.3f}{a:>10.3f}{'예' if a > b else '아니오':>9}")
    print('\n비중이 올랐는데도 회복이 없어야 「어텐션 접근으로는 안 된다」의 근거가 된다.')

In [ ]:
STEPS = ['step6_steer']
# ⑧ 결과 zip으로 묶어 내려받기
import shutil, os, glob

def pack(step):
    d = f'results/{step}'
    if not os.path.isdir(d):
        print(f'  [건너뜀] {d} 폴더가 없다 — 이 스텝은 아직 안 돌렸다')
        return None
    n = len(glob.glob(f'{d}/*.json'))
    if n == 0:
        print(f'  [건너뜀] {d} 가 비어 있다')
        return None
    path = shutil.make_archive(step, 'zip', d)
    print(f'  {step}: {n}개 → {path} ({os.path.getsize(path)/1e6:.1f}MB)')
    return path

print('results/ 안에 있는 폴더:', sorted(os.listdir('results')) if os.path.isdir('results') else '(results 폴더 없음)')
print()
made = [p for p in (pack(s) for s in STEPS) if p]

if not made:
    print('\n내려받을 것이 없다. 실행 셀(⑤)을 먼저 돌렸는지 확인할 것.')
else:
    try:
        from google.colab import files
        for p in made:
            files.download(p)
        print('\n다운로드 시작. 브라우저가 막으면 왼쪽 **파일 탐색기**에서 직접 받으면 된다.')
    except Exception as e:
        print(f'\nColab 자동 다운로드 불가({type(e).__name__}). 위 경로에서 직접 받을 것.')
